## Using U-Net for Binary Segmentation of Bean Bucket XCT Images

Adapted from https://github.com/usuyama/pytorch-unet 

In [ ]:
import os

if not os.path.exists("pytorch_unet.py"):
  if not os.path.exists("pytorch_unet"):
    !git clone https://github.com/usuyama/pytorch-unet.git

  %cd pytorch-unet

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# model.to(device)
# tensors = tensors.to(device)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

IMAGE_DIR = "../BeanBucketImages"
MASK_DIR  = "../BeanBucketMasks_Otsu"
IMG_SIZE  = (256, 256)   # (W, H) for PIL
NUM_PREVIEW = 3
NUM_CLASSES = 2

image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".tif", ".tiff"))])
mask_files  = sorted([f for f in os.listdir(MASK_DIR)  if f.lower().endswith((".tif", ".tiff"))])
assert len(image_files) == len(mask_files) and len(image_files) > 0

def load_img(path):
    return np.array(Image.open(path).convert("RGB").resize(IMG_SIZE))

def load_mask(path):
    # keep as single-channel; nearest resize to preserve labels
    return np.array(Image.open(path).convert("L").resize(IMG_SIZE, Image.NEAREST))

imgs  = [load_img(os.path.join(IMAGE_DIR, f)) for f in image_files[:NUM_PREVIEW]]
masks = [load_mask(os.path.join(MASK_DIR,  f)) for f in mask_files[:NUM_PREVIEW]]

fig, axes = plt.subplots(NUM_PREVIEW, 2, figsize=(7, 3 * NUM_PREVIEW))
for i in range(NUM_PREVIEW):
    axes[i, 0].imshow(imgs[i]);  axes[i, 0].set_title(image_files[i]); axes[i, 0].axis("off")
    axes[i, 1].imshow(masks[i], cmap="gray"); axes[i, 1].set_title(mask_files[i]); axes[i, 1].axis("off")
plt.tight_layout()
plt.show()

print("mask unique values (sample):", np.unique(masks[0]))

In [ ]:
import random
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15
SAMPLE_N   = 100  # number of pairs to randomly select

# ── Dataset class ─────────────────────────────────────────────────────────────
class ImageMaskDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img  = Image.open(self.image_paths[idx]).convert("RGB").resize(IMG_SIZE)
        img  = np.array(img)

        mask = Image.open(self.mask_paths[idx]).convert("L").resize(IMG_SIZE, Image.NEAREST)
        mask = (np.array(mask) > 128).astype(np.uint8)

        one_hot = np.zeros((NUM_CLASSES, IMG_SIZE[0], IMG_SIZE[1]), dtype=np.float32)
        for c in range(NUM_CLASSES):
            one_hot[c] = (mask == c).astype(np.float32)

        if self.transform:
            img = self.transform(img)

        return img, one_hot

# ── Discover files ────────────────────────────────────────────────────────────
all_images = sorted([os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR)])
all_masks  = sorted([os.path.join(MASK_DIR,  f) for f in os.listdir(MASK_DIR)])

assert len(all_images) == len(all_masks), \
    f"Mismatch: {len(all_images)} images vs {len(all_masks)} masks"

# ── Randomly sample 100 pairs ─────────────────────────────────────────────────
random.seed(42)  # remove or change for different random selection each run
indices    = random.sample(range(len(all_images)), SAMPLE_N)
all_images = [all_images[i] for i in indices]
all_masks  = [all_masks[i]  for i in indices]

print(f"Randomly selected {SAMPLE_N} pairs from {len(indices)} total")

# ── Split ─────────────────────────────────────────────────────────────────────
n       = len(all_images)
n_test  = int(n * TEST_SPLIT)
n_val   = int(n * VAL_SPLIT)
n_train = n - n_val - n_test

train_imgs = all_images[:n_train]
val_imgs   = all_images[n_train : n_train + n_val]
test_imgs  = all_images[n_train + n_val:]

train_msks = all_masks[:n_train]
val_msks   = all_masks[n_train : n_train + n_val]
test_msks  = all_masks[n_train + n_val:]

print(f"Total: {n} | Train: {n_train} | Val: {n_val} | Test: {n_test}")

# ── Transforms ────────────────────────────────────────────────────────────────
trans = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Datasets & dataloaders ────────────────────────────────────────────────────
train_set = ImageMaskDataset(train_imgs, train_msks, transform=trans)
val_set   = ImageMaskDataset(val_imgs,   val_msks,   transform=trans)
test_set  = ImageMaskDataset(test_imgs,  test_msks,  transform=trans)

image_datasets = {'train': train_set, 'val': val_set, 'test': test_set}

batch_size = 8

dataloaders = {
    'train': DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=0),
    'val':   DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=0),
    'test':  DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=0),
}

dataset_sizes = {x: len(image_datasets[x]) for x in image_datasets}
print(f"Dataset sizes: {dataset_sizes}")

In [ ]:
import torchvision.utils

def reverse_transform(inp):
  inp = inp.numpy().transpose((1, 2, 0))
  mean = np.array([0.485, 0.456, 0.406])
  std = np.array([0.229, 0.224, 0.225])
  inp = std * inp + mean
  inp = np.clip(inp, 0, 1)
  inp = (inp * 255).astype(np.uint8)

  return inp

# Get a batch of training data
inputs, masks = next(iter(dataloaders['train']))

print(inputs.shape, masks.shape)

plt.imshow(reverse_transform(inputs[3]))

# Define a UNet module

In [ ]:
import torch.nn as nn
import torchvision.models


def convrelu(in_channels, out_channels, kernel, padding):
  return nn.Sequential(
    nn.Conv2d(in_channels, out_channels, kernel, padding=padding),
    nn.ReLU(inplace=True),
  )


class ResNetUNet(nn.Module):
  def __init__(self, n_class):
    super().__init__()

    self.base_model = models.resnet18(weights=None)   # no pretrained download; use weights=models.ResNet18_Weights.DEFAULT if you want pretrained
    self.base_layers = list(self.base_model.children())

    self.layer0 = nn.Sequential(*self.base_layers[:3]) # size=(N, 64, x.H/2, x.W/2)
    self.layer0_1x1 = convrelu(64, 64, 1, 0)
    self.layer1 = nn.Sequential(*self.base_layers[3:5]) # size=(N, 64, x.H/4, x.W/4)
    self.layer1_1x1 = convrelu(64, 64, 1, 0)
    self.layer2 = self.base_layers[5]  # size=(N, 128, x.H/8, x.W/8)
    self.layer2_1x1 = convrelu(128, 128, 1, 0)
    self.layer3 = self.base_layers[6]  # size=(N, 256, x.H/16, x.W/16)
    self.layer3_1x1 = convrelu(256, 256, 1, 0)
    self.layer4 = self.base_layers[7]  # size=(N, 512, x.H/32, x.W/32)
    self.layer4_1x1 = convrelu(512, 512, 1, 0)

    self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

    self.conv_up3 = convrelu(256 + 512, 512, 3, 1)
    self.conv_up2 = convrelu(128 + 512, 256, 3, 1)
    self.conv_up1 = convrelu(64 + 256, 256, 3, 1)
    self.conv_up0 = convrelu(64 + 256, 128, 3, 1)

    self.conv_original_size0 = convrelu(3, 64, 3, 1)
    self.conv_original_size1 = convrelu(64, 64, 3, 1)
    self.conv_original_size2 = convrelu(64 + 128, 64, 3, 1)

    self.conv_last = nn.Conv2d(64, n_class, 1)

  def forward(self, input):
    x_original = self.conv_original_size0(input)
    x_original = self.conv_original_size1(x_original)

    layer0 = self.layer0(input)
    layer1 = self.layer1(layer0)
    layer2 = self.layer2(layer1)
    layer3 = self.layer3(layer2)
    layer4 = self.layer4(layer3)

    layer4 = self.layer4_1x1(layer4)
    x = self.upsample(layer4)
    layer3 = self.layer3_1x1(layer3)
    x = torch.cat([x, layer3], dim=1)
    x = self.conv_up3(x)

    x = self.upsample(x)
    layer2 = self.layer2_1x1(layer2)
    x = torch.cat([x, layer2], dim=1)
    x = self.conv_up2(x)

    x = self.upsample(x)
    layer1 = self.layer1_1x1(layer1)
    x = torch.cat([x, layer1], dim=1)
    x = self.conv_up1(x)

    x = self.upsample(x)
    layer0 = self.layer0_1x1(layer0)
    x = torch.cat([x, layer0], dim=1)
    x = self.conv_up0(x)

    x = self.upsample(x)
    x = torch.cat([x, x_original], dim=1)
    x = self.conv_original_size2(x)

    out = self.conv_last(x)

    return out

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import pytorch_unet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

model = ResNetUNet(NUM_CLASSES)
model = model.to(device)

In [ ]:
model

In [ ]:
from torchsummary import summary
summary(model, input_size=(3, 224, 224))

In [ ]:
from collections import defaultdict
import torch.nn.functional as F
from loss import dice_loss

checkpoint_path = "checkpoint.pth"

def calc_loss(pred, target, metrics, bce_weight=0.5):
    bce = F.binary_cross_entropy_with_logits(pred, target)

    pred = torch.sigmoid(pred)
    dice = dice_loss(pred, target)

    loss = bce * bce_weight + dice * (1 - bce_weight)

    metrics['bce'] += bce.data.cpu().numpy() * target.size(0)
    metrics['dice'] += dice.data.cpu().numpy() * target.size(0)
    metrics['loss'] += loss.data.cpu().numpy() * target.size(0)

    return loss

def print_metrics(metrics, epoch_samples, phase):
    outputs = []
    for k in metrics.keys():
        outputs.append("{}: {:4f}".format(k, metrics[k] / epoch_samples))

    print("{}: {}".format(phase, ", ".join(outputs)))

def train_model(model, optimizer, scheduler, num_epochs=25):
    best_loss = 1e10

    # ← ADD THIS: initialize history before the epoch loop
    history = {
        'train_loss': [], 'val_loss': [],
        'train_bce':  [], 'val_bce':  [],
        'train_dice': [], 'val_dice': [],
    }

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        since = time.time()

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            metrics = defaultdict(float)
            epoch_samples = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = calc_loss(outputs, labels, metrics)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                epoch_samples += inputs.size(0)

            print_metrics(metrics, epoch_samples, phase)
            epoch_loss = metrics['loss'] / epoch_samples
            epoch_bce  = metrics['bce']  / epoch_samples
            epoch_dice = metrics['dice'] / epoch_samples

            # ← ADD THIS: record metrics for this phase/epoch
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_bce'].append(epoch_bce)
            history[f'{phase}_dice'].append(epoch_dice)

            if phase == 'train':
                scheduler.step()
                for param_group in optimizer.param_groups:
                    print("LR", param_group['lr'])

            if phase == 'val' and epoch_loss < best_loss:
                print(f"saving best model to {checkpoint_path}")
                best_loss = epoch_loss
                torch.save(model.state_dict(), checkpoint_path)

        time_elapsed = time.time() - since
        print('{:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))

    print('Best val loss: {:4f}'.format(best_loss))

    model.load_state_dict(torch.load(checkpoint_path))
    return model, history  # ← history is now defined and populated

In [ ]:
import torch
import torch.optim as optim
from torch.optim import lr_scheduler
import time

model = ResNetUNet(NUM_CLASSES).to(device)

# freeze backbone layers
for l in model.base_layers:
  for param in l.parameters():
    param.requires_grad = False

optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=8, gamma=0.1)

model, history = train_model(model, optimizer_ft, exp_lr_scheduler, num_epochs=20)

In [ ]:
# ── Plot training & validation loss / metric curves ───────────────────────
import matplotlib.pyplot as plt

epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training & Validation Metrics', fontsize=14)

metric_cfg = [
    ('Combined loss (BCE + Dice)', 'Combined Loss (BCE + Dice)',  'tab:blue',   'tab:orange'),
    ('Binary cross-entropy (BCE) loss',  'Binary Cross-Entropy Loss',   'tab:green',  'tab:red'),
    ('Dice loss', 'Dice Loss',                   'tab:purple', 'tab:brown'),
]

for ax, (key, title, tc, vc) in zip(axes, metric_cfg):
    ax.plot(epochs, history[f'train_{key}'], color=tc, marker='o', label='Train')
    ax.plot(epochs, history[f'val_{key}'],   color=vc, marker='s', linestyle='--', label='Val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# ── Test set evaluation ───────────────────────────────────────────────────────
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

model.eval()

test_metrics      = defaultdict(float)
test_samples      = 0

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        calc_loss(outputs, labels, test_metrics)
        test_samples += inputs.size(0)

print("=== Test Set Results ===")
print_metrics(test_metrics, test_samples, "test")

In [ ]:
# ── Visual predictions on test set ───────────────────────────────────────────
NUM_VISUAL = 4  # how many samples to show

def reverse_transform(inp):
    inp   = inp.numpy().transpose((1, 2, 0))
    mean  = np.array([0.485, 0.456, 0.406])
    std   = np.array([0.229, 0.224, 0.225])
    inp   = std * inp + mean
    inp   = np.clip(inp, 0, 1)
    return (inp * 255).astype(np.uint8)

def masks_to_colorimg(mask):
    colors = np.array([
        [0,   0,   0],    # class 0 — background (black)
        [255, 255, 255],  # class 1 — foreground (white)
    ], dtype=np.uint8)
    return colors[np.argmax(mask, axis=0)]

model.eval()

# grab one batch from the test loader
inputs, labels = next(iter(dataloaders['test']))
inputs = inputs.to(device)

with torch.no_grad():
    preds = model(inputs)
    preds = torch.sigmoid(preds).cpu().numpy()

inputs = inputs.cpu()
labels = labels.numpy()

n_show = min(NUM_VISUAL, inputs.size(0))
fig, axes = plt.subplots(n_show, 3, figsize=(10, n_show * 3))
fig.suptitle("Test Predictions\n(Left: Image | Middle: Ground Truth | Right: Prediction)", fontsize=12)

for i in range(n_show):
    axes[i, 0].imshow(reverse_transform(inputs[i]))
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masks_to_colorimg(labels[i]))
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(masks_to_colorimg(preds[i]))
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

## Next steps

Try tweaking the hyper-parameters for better accuracy e.g.

- learning rates and schedules
- loss weights
- unfreezing layers
- batch size
- etc.